# ☸️ Chapter 12: Container Orchestration: Kubernetes

📖 Book: Chapter 12, pp. 466–511 · ⏱️ 3 hours · 🧱 Needs: Ch 11 · Level: Advanced

---

> **🎯 Key idea:** Kubernetes (`apps/v1` Deployments, Services, Probes, and Helm 3) schedules and self-heals containers across multi-node clusters.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** Helm v2 (`helm init`, `--name`, `stable/*`)  
> **🆕 Now (2026):** Helm 3 (`prometheus-community`, no `--name`)  
> **Why:** `stable` repo deprecated; Tiller removed

> **📜 Then (book, ~2020):** `minikube` v1.15 + `kompose`  
> **🆕 Now (2026):** `kind` or `minikube` (`kompose` bootstrap-only)  
> **Why:** `kind` runs K8s fast inside Docker

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
A single Docker host goes down for kernel patching, taking your entire web app offline with it. You need automatic rescheduling across multiple nodes, zero-downtime rolling updates, and health-gated traffic. Kubernetes (`apps/v1` Deployments, Services, Probes, and Helm 3) automates cluster-wide self-healing.

### Concept: Deployments, Services & Probes
**What:** Declarative `apps/v1` + `v1` objects  
**Why:** Self-healing replicas & zero-downtime rollouts  
**How:** `readinessProbe` gates traffic; `livenessProbe` restarts  
**When:** Any multi-node container workload

📖 *Book listing:* `kompose` Conversion & `initContainer` Migrations (pp. 470–485)

**Plain-language analogy:** If Docker is a single shipping container, Kubernetes is the port crane operator that places containers onto ships (Nodes), replaces broken ones (Pods), and directs trucks (Services).

**Walkthrough:** The book converts `docker-compose.yaml` with `kompose convert`, moves DB migrations into an `initContainer` inside the Deployment, and exposes `app` via a `LoadBalancer` Service.

### Concept: Helm 3 Charts
**What:** Templated K8s package manager  
**Why:** Installs Prometheus/Grafana cleanly  
**How:** `helm repo add` + `helm install <name> <chart>`  
**When:** Standard cluster add-ons

📖 *Book listing:* Helm Prometheus & Grafana Monitoring Charts (pp. 495–509)

**Plain-language analogy:** A `readinessProbe` is an 'Open' sign on a shop door (don't send customers until cash registers are ready); a `livenessProbe` restarts the shop if the manager faints.

**Walkthrough:** The book installs Prometheus and Grafana into a `monitoring` namespace and forwards ports locally. Modern Helm 3 drops Tiller and `--name` and uses `prometheus-community`.

## 6. 🟢 Beginner Examples

#### Example 1: Beginner Example 1: Build an apps/v1 Deployment manifest dict
*Expected output:* Printed labelled output below.

In [1]:
# Beginner Example 1: Build an apps/v1 Deployment manifest dict
def make_deployment(name: str, image: str, replicas: int = 2, port: int = 5000) -> dict:
    return {
        'apiVersion': 'apps/v1',
        'kind': 'Deployment',
        'metadata': {'name': name, 'labels': {'app': name}},
        'spec': {
            'replicas': replicas,
            'selector': {'matchLabels': {'app': name}},
            'template': {
                'metadata': {'labels': {'app': name}},
                'spec': {
                    'containers': [{
                        'name': name,
                        'image': image,
                        'ports': [{'containerPort': port}],
                        'resources': {'limits': {'cpu': '500m', 'memory': '256Mi'}},
                        'readinessProbe': {'httpGet': {'path': '/health', 'port': port}},
                    }]
                },
            },
        },
    }

dep = make_deployment('flask-api', 'myuser/flask-api:1.0.0')
print('Created Deployment:', dep['metadata']['name'], 'replicas=', dep['spec']['replicas'])

Created Deployment: flask-api replicas= 2


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 2: Beginner Example 2: Build a v1 Service manifest
*Expected output:* Printed labelled output below.

In [2]:
# Beginner Example 2: Build a v1 Service manifest
def make_service(name: str, port: int = 80, target_port: int = 5000, svc_type: str = 'ClusterIP') -> dict:
    return {
        'apiVersion': 'v1',
        'kind': 'Service',
        'metadata': {'name': name},
        'spec': {'type': svc_type, 'selector': {'app': name}, 'ports': [{'port': port, 'targetPort': target_port}]},
    }

svc = make_service('flask-api')
print('Service selector:', svc['spec']['selector'])

Service selector: {'app': 'flask-api'}


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 3: Beginner Example 3: Parse Kubernetes YAML via yaml.safe_load
*Expected output:* Printed labelled output below.

In [3]:
# Beginner Example 3: Parse Kubernetes YAML via yaml.safe_load
K8S_YAML = '''
apiVersion: apps/v1
kind: Deployment
metadata:
  name: worker
spec:
  replicas: 3
'''
try:
    import yaml
    print('Parsed YAML:', yaml.safe_load(K8S_YAML))
except ImportError:
    print('PyYAML not installed')

Parsed YAML: {'apiVersion': 'apps/v1', 'kind': 'Deployment', 'metadata': {'name': 'worker'}, 'spec': {'replicas': 3}}


**What the output means:** Shows the exact exception caught cleanly instead of crashing the whole script.

> **🧪 Try it:** Change the invalid input to a valid one and observe the `except` branch no longer triggers.

## 7. 🟡 Intermediate Examples: Policy Validator & `kubernetes` Client

#### Example 4: Intermediate Example 1: Validate Deployment production readiness
*Expected output:* Printed labelled output below.

In [4]:
# Intermediate Example 1: Validate Deployment production readiness
def validate_k8s_deployment(manifest: dict) -> list[str]:
    issues: list[str] = []
    if manifest.get('kind') != 'Deployment':
        return ['Not a Deployment']
    spec = manifest.get('spec', {})
    if spec.get('replicas', 1) < 2:
        issues.append('replicas < 2 (not HA)')
    for c in spec.get('template', {}).get('spec', {}).get('containers', []):
        img = c.get('image', '')
        if ':latest' in img or ':' not in img:
            issues.append(f"{c.get('name')}: avoid :latest image tag")
        if 'limits' not in c.get('resources', {}):
            issues.append(f"{c.get('name')}: missing resources.limits")
        if 'readinessProbe' not in c:
            issues.append(f"{c.get('name')}: missing readinessProbe")
    return issues

print('Good manifest issues:', validate_k8s_deployment(dep))

Good manifest issues: []


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

> **⚠️ Warning:** Cloud commands incur costs—clean up test resources afterward.

#### Example 5: Intermediate Example 2 (External + Mock Fallback): kubernetes Python client
*Expected output:* Simulated fallback if optional package is missing.

In [5]:
# Intermediate Example 2 (External + Mock Fallback): kubernetes Python client
# Requires: pip install kubernetes
# WARNING: Cloud clusters (GKE/EKS) incur hourly charges. Destroy test clusters when finished!
try:
    from kubernetes import client
    print('kubernetes client available')
except ImportError:
    mock_pods = [{'namespace': 'default', 'name': 'flask-api-7d9', 'status': 'Running'}]
    print('kubernetes package not installed; simulated pods:', mock_pods)

kubernetes package not installed; simulated pods: [{'namespace': 'default', 'name': 'flask-api-7d9', 'status': 'Running'}]


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **Pod Status Triage Helper** mapping K8s failure states to `kubectl` remediation commands.

#### Example 6: REMEDIATION = {
*Expected output:* Printed labelled output below.

In [6]:
REMEDIATION = {
    'CrashLoopBackOff': 'kubectl logs <pod> --previous',
    'ImagePullBackOff': 'Verify image tag and imagePullSecrets',
    'Pending': 'kubectl describe pod <pod> (check CPU/RAM or unbound PVC)',
}

def triage_pods(pods: list[dict]) -> list[dict]:
    return [{'pod': p['name'], 'state': p['state'], 'action': REMEDIATION.get(p['state'], 'OK')} for p in pods if p['state'] != 'Running']

print(triage_pods([{'name': 'api-1', 'state': 'Running'}, {'name': 'db-0', 'state': 'CrashLoopBackOff'}]))

[{'pod': 'db-0', 'state': 'CrashLoopBackOff', 'action': 'kubectl logs <pod> --previous'}]


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 7: Real Tool Demo: kubectl client & helm 3 check (guarded with shutil.which)
*Expected output:* Printed labelled output below.

In [7]:
# Real Tool Demo: kubectl --dry-run=client & helm 3 check
import json, shutil, subprocess, tempfile
from pathlib import Path
if shutil.which('kubectl'):
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp) / 'svc.json'
        p.write_text(json.dumps(make_service('demo')), encoding='utf-8')
        cp = subprocess.run(['kubectl', 'apply', '--dry-run=client', '-f', str(p)], capture_output=True, text=True)
        print('kubectl dry-run:', (cp.stdout or cp.stderr).strip())
if shutil.which('helm'):
    print('helm:', subprocess.run(['helm', 'version', '--short'], capture_output=True, text=True).stdout.strip())

kubectl dry-run: error: error validating "/tmp/tmpc8gs7s9s/svc.json": error validating data: failed to download openapi: Get "https://192.168.49.2:8443/openapi/v2?timeout=32s": dial tcp 192.168.49.2:8443: i/o timeout; if you choose to ignore these errors, turn validation off with --validate=false
helm: v3.21.0+ge0878d4


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **Using `:latest` image tags** -> Nodes serve stale cached images. ✅ Pin immutable SemVer/SHA tags.
- ❌ **Missing `resources.limits`** -> One Pod OOMs entire worker node. ✅ Set `cpu` & `memory` limits.
- ❌ **GKE PVC `lost+found` crash** -> Postgres `initdb` fails (`CrashLoopBackOff`). ✅ Delete `lost+found/` via temp `busybox` pod.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement extract_images and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [8]:
# Starter Exercise Cell (implement extract_images and run to verify!)
def _run_starter_checks():
    assert extract_images(dep) == ['myuser/flask-api:1.0.0']
def extract_images(dep: dict) -> list[str]:
    raise NotImplementedError('Implement extract_images')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement extract_images


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** How do a Pod and a Deployment differ?

<details>
<summary>✅ Answer 1</summary>

A Pod is a single running instance; a Deployment manages Pod replicas and rolling updates.

</details>

**Q2:** How does a Service route traffic to Pods?

<details>
<summary>✅ Answer 2</summary>

Via label selectors (`spec.selector` matching Pod `metadata.labels`).

</details>

**Q3:** What is an `initContainer` used for?

<details>
<summary>✅ Answer 3</summary>

Running setup tasks (like DB migrations) to completion before the main container starts.

</details>

**Q4:** How do `readinessProbe` and `livenessProbe` differ?

<details>
<summary>✅ Answer 4</summary>

Readiness gates Service traffic; Liveness restarts a stuck container.

</details>

**Q5:** What changed in Helm 3 vs. Helm 2?

<details>
<summary>✅ Answer 5</summary>

Tiller was removed, `--name` is positional (`helm install <name> <chart>`), and `stable/*` was replaced by repos like `prometheus-community`.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Challenge**: Collect `dep_apps` from Deployment labels, then check every Service selector.

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [9]:
# Complete Solutions + Assert Verification
def extract_images(dep: dict) -> list[str]:
    return [c['image'] for c in dep['spec']['template']['spec']['containers']]

def merge_helm_values(base: dict, override: dict) -> dict:
    return {**base, **override}

def find_unhealthy_pods(pods: list[dict]) -> list[str]:
    return [p['name'] for p in pods if not p.get('ready', True) or p.get('restarts', 0) > 3]

def admission_check(manifests: list[dict]) -> dict:
    errors: list[str] = []
    dep_apps: set[str] = set()
    for m in manifests:
        if m.get('kind') == 'Deployment':
            errors.extend(validate_k8s_deployment(m))
            if lbl := m.get('metadata', {}).get('labels', {}).get('app'):
                dep_apps.add(lbl)
    for m in manifests:
        if m.get('kind') == 'Service':
            sel = m.get('spec', {}).get('selector', {}).get('app')
            if sel not in dep_apps:
                errors.append(f'Service selector {sel!r} matches no Deployment')
    return {'admitted': len(errors) == 0, 'errors': errors}

assert extract_images(dep) == ['myuser/flask-api:1.0.0']
assert merge_helm_values({'a': 1}, {'a': 2}) == {'a': 2}
assert find_unhealthy_pods([{'name': 'p1', 'ready': True, 'restarts': 5}]) == ['p1']
assert admission_check([dep, svc])['admitted'] is True
print('All Chapter 12 assertions PASSED!')

All Chapter 12 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `kubectl get pods,svc` — Inspect cluster workloads (e.g. `kubectl logs <pod> --previous`)
- `helm install prometheus ...` — Install Helm 3 chart (e.g. `helm install prometheus prometheus-community/prometheus -n monitoring --create-namespace`)
- `kubectl apply --dry-run=client` — Validate YAML locally (e.g. `kubectl apply -f dep.yaml --dry-run=client`)

### ✅ What You Can Now Do
- Build and validate `apps/v1` Deployments and `v1` Services (HA replicas, pinned tags, limits, probes).
- Triage `CrashLoopBackOff`, `ImagePullBackOff`, and `Pending` Pods to `kubectl` actions.
- Dry-run manifests with `kubectl apply --dry-run=client` and use Helm 3 (`prometheus-community`).

### ➡️ Next Chapter
Chapter 13 builds on this by running event-driven code without managing clusters at all using Serverless FaaS and CDK v2.